<a href="https://colab.research.google.com/github/imkmaurya/Insurance_premium_predictor/blob/main/insurance_premium_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [428]:
import numpy as np
import  pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

In [429]:
df=pd.read_csv("/content/synthetic_insurance_with_target (1).csv")

In [430]:
df.head()

,age,height,weight,income_lpa,smoker,city,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,34,1.89,49.9,8.87,True,Kolkata,business_owner,13.97,adult,medium,3,medium
1,47,1.66,94.4,19.71,True,Pune,freelancer,34.26,middle_aged,high,3,high
2,60,1.53,87.9,8.29,False,Mumbai,unemployed,37.55,senior,medium,1,medium
3,69,1.99,66.1,6.87,False,Chandigarh,unemployed,16.69,senior,low,2,low
4,24,1.53,57.0,27.64,False,Hyderabad,retired,24.35,young,low,3,low


In [431]:
df_feat=df.copy()

In [432]:
df['occupation'].unique()

array(['business_owner', 'freelancer', 'unemployed', 'retired', 'student',
       'private_job', 'government_job'], dtype=object)

In [433]:
df["bmi"]=df["weight"]/(df["height"])**2

In [434]:
def change_age(age):
  if age < 25:
    return "young"
  elif age < 45:
    return "adult"
  else:
    return "middle_aged"

  return "senior"

In [435]:
df["age_level"]=df["age"].apply(change_age)

In [436]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]


In [437]:
def newcities(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [438]:
df["city_tier"]=df['city'].apply(newcities)

In [439]:
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [440]:
df["lifestyle_risk"]=df.apply(lifestyle_risk,axis=1)

In [441]:
df.head()

,age,height,weight,income_lpa,smoker,city,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category,age_level
0,34,1.89,49.9,8.87,True,Kolkata,business_owner,13.969374,adult,medium,1,medium,adult
1,47,1.66,94.4,19.71,True,Pune,freelancer,34.257512,middle_aged,high,1,high,middle_aged
2,60,1.53,87.9,8.29,False,Mumbai,unemployed,37.549660,senior,medium,1,medium,middle_aged
3,69,1.99,66.1,6.87,False,Chandigarh,unemployed,16.691498,senior,low,2,low,middle_aged
4,24,1.53,57.0,27.64,False,Hyderabad,retired,24.349609,young,low,1,low,young


In [442]:
df=df.drop(columns=["height","weight","smoker","age","city"])

In [443]:
df.head()

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category,age_level
0,8.87,business_owner,13.969374,adult,medium,1,medium,adult
1,19.71,freelancer,34.257512,middle_aged,high,1,high,middle_aged
2,8.29,unemployed,37.549660,senior,medium,1,medium,middle_aged
3,6.87,unemployed,16.691498,senior,low,2,low,middle_aged
4,27.64,retired,24.349609,young,low,1,low,young


In [444]:
X=df.drop(columns="insurance_premium_category")
y=df["insurance_premium_category"]

In [445]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [446]:
cato_col=["occupation","lifestyle_risk","age_level"]
num_col=["income_lpa","bmi"]

In [447]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [448]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), cato_col),
        ("num", "passthrough", num_col)
    ]
)

In [449]:
pipe=Pipeline([
    ("preprocessor",preprocessor),
    ("model",RandomForestClassifier(random_state=1))
]
)

In [450]:
from sklearn.metrics import accuracy_score

In [451]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['occupation',
                                                   'lifestyle_risk',
                                                   'age_level']),
                                                 ('num', 'passthrough',
                                                  ['income_lpa', 'bmi'])])),
                ('model', RandomForestClassifier(random_state=1))])

In [452]:
y_pred=pipe.predict(X_test)

In [453]:
accuracy_score(y_test,y_pred)

0.88

In [454]:
import pickle
with open("pipe.pkl","wb") as f:
  pickle.dump(pipe,f)
